# Takwim: timetable solver (Person 2)

Pipeline: **load → sessions → assign professors → allowed rooms/times → CP-SAT model (hard + soft rules) → solve → check → export**.

Change `SIZE` in the first cell to switch between the small and the real department.

In [ ]:
import json
import os
from collections import Counter, defaultdict

from ortools.sat.python import cp_model
from groups import build_groups, session_demand

SIZE = "real"                     # "small" for quick tests, "real" for the full department
TIME_LIMIT = 60                   # seconds for the timetable solve
WORKERS = 8                       # CPU threads (check yours with: nproc)

dept = json.load(open(f"out/department_{SIZE}.json"))
groups = build_groups(dept)

group_by_id = {g["id"]: g for g in groups}
level_by_id = {l["id"]: l for l in dept["levels"]}
days = dept["calendar"]["days"]
hours = dept["rules"]["session_minutes"] / 60

## 1. Sessions to place
One lecture per module for each section, one TD/TP per module for each group.

In [ ]:
sessions = [{"id": f"s{i:03d}", "module": m, "kind": k, "group": g}
            for i, (m, k, g) in enumerate(session_demand(dept, groups))]
print(len(sessions), "sessions")

## 2. Assign a professor to each session
A small CP-SAT model: each session gets one professor who asked for it (vœux), nobody goes over their weekly hours, and vœu ranks are kept as low as possible.

In [ ]:
def candidates(s):
    return [(v["rank"], p["id"])
            for p in dept["professors"] for v in p["voeux"]
            if v["module"] == s["module"] and s["kind"] in v["kinds"]]

amodel = cp_model.CpModel()
y = {}                                   # (session, prof) -> does this prof teach it?
cost = []
for s in sessions:
    for rank, p in candidates(s):
        y[s["id"], p] = amodel.NewBoolVar(f"{s['id']}_{p}")
        cost.append(rank * y[s["id"], p])
    amodel.AddExactlyOne(y[s["id"], p] for _, p in candidates(s))

for p in dept["professors"]:
    mine = [v for (sid, pid), v in y.items() if pid == p["id"]]
    if mine:
        amodel.Add(sum(mine) <= round(p["max_hours"] / hours))

amodel.Minimize(sum(cost))
asolver = cp_model.CpSolver()
asolver.parameters.max_time_in_seconds = 10
astatus = asolver.Solve(amodel)
print("assignment:", asolver.StatusName(astatus))

for s in sessions:
    for _, p in candidates(s):
        if asolver.Value(y[s["id"], p]) == 1:
            s["prof"] = p

## 3. Allowed rooms and times
Only rooms of the right type that are big enough, and only times on the level's days when the professor is free. Forbidden combinations never become variables.

In [ ]:
times = [(d, sl["id"]) for d in days for sl in dept["calendar"]["slots"]]
blocked = {(u["prof"], u["day"], u["slot"]) for u in dept["unavailability"]}
room_types = dept["rules"]["room_types_for"]

for s in sessions:
    g = group_by_id[s["group"]]
    level = level_by_id[g["level"]]
    s["rooms"] = [r["id"] for r in dept["rooms"]
                  if r["type"] in room_types[s["kind"]] and r["capacity"] >= g["size"]]
    s["times"] = [t for t, (d, sl) in enumerate(times)
                  if d in level["days"] and (s["prof"], d, sl) not in blocked]

print(len(times), "time slots,", sum(len(s["rooms"]) * len(s["times"]) for s in sessions), "possible placements")

## 4. Variables
`x[session, time, room]` = 1 if the session is at that time in that room. The `by_...` boxes group the variables for the rules.

In [ ]:
model = cp_model.CpModel()
x = {}
by_session = defaultdict(list)
by_room = defaultdict(list)
by_prof = defaultdict(list)
by_group = defaultdict(list)

for s in sessions:
    for t in s["times"]:
        for r in s["rooms"]:
            v = model.NewBoolVar(f"{s['id']}_t{t}_{r}")
            x[s["id"], t, r] = v
            by_session[s["id"]].append(v)
            by_room[t, r].append(v)
            by_prof[t, s["prof"]].append(v)
            by_group[t, s["group"]].append(v)

print(len(x), "variables")

## 5. Hard rules (never broken)

In [ ]:
# every session placed exactly once
for s in sessions:
    model.AddExactlyOne(by_session[s["id"]])

# a room, a professor: at most one session at a time
for vs in by_room.values():
    model.AddAtMostOne(vs)
for vs in by_prof.values():
    model.AddAtMostOne(vs)

# a group: at most one session at a time, counting its section's lectures
for g in groups:
    if g["kind"] == "group":
        for t in range(len(times)):
            vs = by_group[t, g["id"]] + by_group[t, g["parent"]]
            if len(vs) > 1:
                model.AddAtMostOne(vs)

# daily limits for groups and professors
max_group = dept["rules"]["max_sessions_per_day_group"]
max_prof = dept["rules"]["max_sessions_per_day_prof"]
for d in days:
    day_times = [t for t, (day, sl) in enumerate(times) if day == d]
    for g in groups:
        if g["kind"] == "group":
            vs = [v for t in day_times for v in by_group[t, g["id"]] + by_group[t, g["parent"]]]
            if vs:
                model.Add(sum(vs) <= max_group)
    for p in dept["professors"]:
        vs = [v for t in day_times for v in by_prof[t, p["id"]]]
        if vs:
            model.Add(sum(vs) <= max_prof)

## 6. Soft rules (penalties to minimise)
| Rule | Weight |
|---|---|
| gap in a group's day | 5 |
| session on Saturday | 5 |
| day with only one session | 4 |
| day that doesn't start at 08:00 | 2 |
| TD/TP before its lecture on the same day | 2 |
| session in the last slot (15:30) | 1 |

In [ ]:
W = {"gap": 5, "saturday": 5, "single_day": 4, "late_start": 2, "td_before_lecture": 2, "late": 1}
penalties = []

# start time and day number of each session
time_of = defaultdict(list)
for (sid, t, r), v in x.items():
    time_of[sid].append((t, v))
slots_per_day = len(dept["calendar"]["slots"])
start, day_of = {}, {}
for s in sessions:
    start[s["id"]] = model.NewIntVar(0, len(times) - 1, f"start_{s['id']}")
    model.Add(start[s["id"]] == sum(t * v for t, v in time_of[s["id"]]))
    day_of[s["id"]] = model.NewIntVar(0, len(days) - 1, f"day_{s['id']}")
    model.AddDivisionEquality(day_of[s["id"]], start[s["id"]], slots_per_day)

# Saturday and the last slot
avoid = set(dept["rules"]["avoid_days"])
last_slot = slots_per_day - 1
for (sid, t, r), v in x.items():
    day, slot = times[t]
    if day in avoid:
        penalties.append(W["saturday"] * v)
    if slot == last_slot:
        penalties.append(W["late"] * v)

# TD/TP before its lecture on the same day
for g in groups:
    if g["kind"] != "group":
        continue
    for s in sessions:
        if s["group"] == g["id"] and s["kind"] in ("td", "tp"):
            for l in sessions:
                if l["group"] == g["parent"] and l["kind"] == "lecture" and l["module"] == s["module"]:
                    other_day = model.NewBoolVar("")
                    after = model.NewBoolVar("")
                    model.Add(day_of[s["id"]] != day_of[l["id"]]).OnlyEnforceIf(other_day)
                    model.Add(start[s["id"]] > start[l["id"]]).OnlyEnforceIf(after)
                    ok = model.NewBoolVar("")
                    model.AddBoolOr([other_day, after]).OnlyEnforceIf(ok)
                    penalties.append(W["td_before_lecture"] * (1 - ok))

# per group and per day: gaps, single-session days, late starts
for g in groups:
    if g["kind"] != "group":
        continue
    for d in level_by_id[g["level"]]["days"]:
        day_times = [t for t, (day, sl) in enumerate(times) if day == d]
        occ = [sum(by_group[t, g["id"]] + by_group[t, g["parent"]]) for t in day_times]
        n = len(day_times)

        for i in range(1, n - 1):                      # gap = something before AND after, nothing now
            before = model.NewBoolVar("")
            after = model.NewBoolVar("")
            for j in range(i):
                model.Add(before >= occ[j])
            for j in range(i + 1, n):
                model.Add(after >= occ[j])
            gap = model.NewBoolVar("")
            model.Add(gap >= before + after - 1 - occ[i])
            penalties.append(W["gap"] * gap)

        has_any = model.NewBoolVar("")                 # does the group come in that day?
        for o in occ:
            model.Add(has_any >= o)
        single = model.NewBoolVar("")                  # exactly one session that day
        model.Add(single >= 2 * has_any - sum(occ))
        penalties.append(W["single_day"] * single)
        late_start = model.NewBoolVar("")              # comes in, but nothing at 08:00
        model.Add(late_start >= has_any - occ[0])
        penalties.append(W["late_start"] * late_start)

model.Minimize(sum(penalties))
print(len(penalties), "penalty terms")

## 7. Solve

In [ ]:
solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = TIME_LIMIT
solver.parameters.num_workers = WORKERS
status = solver.Solve(model)

print(solver.StatusName(status), "in", round(solver.WallTime(), 2), "seconds")
if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print("penalty score:", solver.ObjectiveValue(), "| best possible >=", solver.BestObjectiveBound())

## 8. Read the timetable and save it

In [ ]:
session_by_id = {s["id"]: s for s in sessions}
result = []
if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    for (sid, t, r), v in x.items():
        if solver.Value(v) == 1:
            s = session_by_id[sid]
            day, slot = times[t]
            result.append({"id": sid, "module": s["module"], "kind": s["kind"],
                           "group": s["group"], "prof": s["prof"], "room": r,
                           "day": day, "slot": slot, "status": "scheduled"})

with open(f"out/timetable_{SIZE}.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)
print(len(result), "sessions placed, saved", f"out/timetable_{SIZE}.json")

## 9. Check every hard rule again, without the solver

In [ ]:
def count_clashes(result):
    clashes = 0
    for key in ("room", "prof"):
        c = Counter((r["day"], r["slot"], r[key]) for r in result)
        clashes += sum(n - 1 for n in c.values() if n > 1)
    for g in groups:
        if g["kind"] == "group":
            c = Counter((r["day"], r["slot"]) for r in result if r["group"] in (g["id"], g["parent"]))
            clashes += sum(n - 1 for n in c.values() if n > 1)
    return clashes

def validate(result):
    problems = []
    room_by_id = {r["id"]: r for r in dept["rooms"]}
    if sorted(r["id"] for r in result) != sorted(s["id"] for s in sessions):
        problems.append("some sessions are missing or placed twice")
    for r in result:
        g = group_by_id[r["group"]]
        room = room_by_id[r["room"]]
        if room["type"] not in room_types[r["kind"]]:
            problems.append(f'{r["id"]}: wrong room type')
        if room["capacity"] < g["size"]:
            problems.append(f'{r["id"]}: room too small')
        if (r["prof"], r["day"], r["slot"]) in blocked:
            problems.append(f'{r["id"]}: professor unavailable')
        if r["day"] not in level_by_id[g["level"]]["days"]:
            problems.append(f'{r["id"]}: day not allowed for {g["level"]}')
    for d in days:
        for g in groups:
            if g["kind"] == "group":
                n = sum(1 for r in result if r["day"] == d and r["group"] in (g["id"], g["parent"]))
                if n > max_group:
                    problems.append(f'{g["id"]}: {n} sessions on {d}')
        for p in dept["professors"]:
            n = sum(1 for r in result if r["day"] == d and r["prof"] == p["id"])
            if n > max_prof:
                problems.append(f'{p["id"]}: {n} sessions on {d}')
    for p in dept["professors"]:
        n = sum(1 for r in result if r["prof"] == p["id"])
        if n * hours > p["max_hours"]:
            problems.append(f'{p["id"]}: {n * hours} h > {p["max_hours"]} h')
    return problems

problems = validate(result)
print("clashes:", count_clashes(result))
print("rule violations:", len(problems))
for p in problems[:10]:
    print(" -", p)

## 10. Quality of the timetable

In [ ]:
def evaluate(result):
    report = {"gaps": 0, "late_start": 0, "single_days": 0, "saturday": 0, "late": 0,
              "td_before_lecture_same_day": 0}
    for g in groups:
        if g["kind"] != "group":
            continue
        mine = [r for r in result if r["group"] in (g["id"], g["parent"])]
        for d in days:
            used = sorted(r["slot"] for r in mine if r["day"] == d)
            if used:
                report["gaps"] += (used[-1] - used[0] + 1) - len(used)
                report["late_start"] += used[0]
                if len(used) == 1:
                    report["single_days"] += 1
        for r in mine:
            if r["kind"] in ("td", "tp"):
                lectures = [l for l in mine if l["kind"] == "lecture" and l["module"] == r["module"]]
                if lectures and r["day"] == lectures[0]["day"] and r["slot"] < lectures[0]["slot"]:
                    report["td_before_lecture_same_day"] += 1
    report["saturday"] = sum(1 for r in result if r["day"] in avoid)
    report["late"] = sum(1 for r in result if r["slot"] == last_slot)
    return report

for key, value in evaluate(result).items():
    print(f"{key:28s} {value}")

## 11. Excel export
One file with a sheet per group, and one file per group in `out/timetables_<SIZE>/`.

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill

module_name = {m["id"]: m["name"] for m in dept["modules"]}
prof_name = {p["id"]: p["name"] for p in dept["professors"]}
room_name = {r["id"]: r["name"] for r in dept["rooms"]}
slots = dept["calendar"]["slots"]
kind_fr = {"lecture": "Cours", "td": "TD", "tp": "TP"}
colour = {"lecture": "FFF2CC", "td": "DDEBF7", "tp": "E2EFDA"}

def fill_sheet(ws, g):
    ws.append([g["name"]] + [f'{s["start"]}-{s["end"]}' for s in slots])
    for c in ws[1]:
        c.font = Font(bold=True)
    mine = [r for r in result if r["group"] in (g["id"], g["parent"])]
    for d in days:
        row = [d]
        for s in slots:
            here = [r for r in mine if r["day"] == d and r["slot"] == s["id"]]
            row.append("\n".join(f'{kind_fr[r["kind"]]} {module_name[r["module"]]}\n'
                                 f'{room_name[r["room"]]} · {prof_name[r["prof"]]}' for r in here))
        ws.append(row)
        for col, s in enumerate(slots, start=2):
            here = [r for r in mine if r["day"] == d and r["slot"] == s["id"]]
            if here:
                ws.cell(ws.max_row, col).fill = PatternFill("solid", fgColor=colour[here[0]["kind"]])
    ws.column_dimensions["A"].width = 12
    for col in "BCDEFG":
        ws.column_dimensions[col].width = 28
    for r in ws.iter_rows(min_row=2):
        ws.row_dimensions[r[0].row].height = 60
        for c in r:
            c.alignment = Alignment(wrap_text=True, vertical="top")

all_wb = Workbook()
all_wb.remove(all_wb.active)
folder = f"out/timetables_{SIZE}"
os.makedirs(folder, exist_ok=True)
for g in groups:
    if g["kind"] != "group":
        continue
    name = g["name"].replace(" · ", " ")[:31]
    fill_sheet(all_wb.create_sheet(name), g)
    wb = Workbook()
    wb.active.title = name
    fill_sheet(wb.active, g)
    wb.save(f"{folder}/{g['id']}.xlsx")
all_wb.save(f"out/timetable_{SIZE}.xlsx")
print(f"saved out/timetable_{SIZE}.xlsx and {len(all_wb.sheetnames)} files in {folder}/")